In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
#for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
! git clone https://github.com/huggingface/transformers.git
! pip3 install av
    
import os
import random
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import shutil
import av
import sys
import torch
from tqdm import tqdm
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import VivitConfig, VivitModel, VivitImageProcessor, VivitForVideoClassification

Cloning into 'transformers'...
remote: Enumerating objects: 234292, done.
remote: Counting objects: 100% (26657/26657), done.
remote: Compressing objects: 100% (2097/2097), done.
remote: Total 234292 (delta 25947), reused 24688 (delta 24530), pack-reused 207635 (from 1)
Receiving objects: 100% (234292/234292), 239.58 MiB | 18.98 MiB/s, done.
Resolving deltas: 100% (171792/171792), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.1/33.1 MB 51.0 MB/s eta 0:00:00


In [3]:
data_dir = '../input/hand-wash-dataset/HandWashDataset/HandWashDataset'

classes = ['Step_1', 'Step_2_Left', 'Step_2_Right', 'Step_3', 'Step_4_Left', 'Step_4_Right', 'Step_5_Left',
           'Step_5_Right', 'Step_6_Left', 'Step_6_Right', 'Step_7_Left', 'Step_7_Right']

def get_classes_filenames(data_dir, classes):
    """
    Filenames of files in data_dir/classes[i] for each i
    Args:
        data_dir (`str`): filepath to a directory
        classes (`list[str]`): list of subdirectories of data_dir
    Return:
        filenames (`dict`): dictionary of filenames
    """
    filenames = {}
    for class_name in classes:
        class_dir = os.path.join(data_dir, class_name)
        filenames[class_name] = os.listdir(class_dir)
    return filenames

filenames = get_classes_filenames(data_dir, classes)

In [4]:
# Auxilliary functions from Hugging Face

def read_video_pyav(container, indices):
    '''
    Decode the video with PyAV decoder.
    Args:
        container (`av.container.input.InputContainer`): PyAV container.
        indices (`List[int]`): List of frame indices to decode.
    Returns:
        result (np.ndarray): np array of decoded frames of shape (num_frames, height, width, 3).
    '''
    frames = []
    container.seek(0)
    start_index = indices[0]
    end_index = indices[-1]
    for i, frame in enumerate(container.decode(video=0)):
        if i > end_index:
            break
        if i >= start_index and i in indices:
            frames.append(frame)
    return np.stack([x.to_ndarray(format="rgb24") for x in frames])

def sample_frame_indices(clip_len, frame_sample_rate, seg_len):
    '''
    Sample a given number of frame indices from the video.
    Args:
        clip_len (`int`): Total number of frames to sample.
        frame_sample_rate (`int`): Sample every n-th frame.
        seg_len (`int`): Maximum allowed index of sample's last frame.
    Returns:
        indices (`List[int]`): List of sampled frame indices
    '''
    converted_len = int(clip_len * frame_sample_rate)
    end_idx = np.random.randint(converted_len, seg_len)
    start_idx = end_idx - converted_len
    indices = np.linspace(start_idx, end_idx, num=clip_len)
    indices = np.clip(indices, start_idx, end_idx - 1).astype(np.int64)
    return indices


label2id = {class_name: idx for idx, class_name in enumerate(classes)}

# Class of dataset
class HandwashingDataset(Dataset):
    """Handwashing Dataset."""
    def __init__(self, file_path, file_names, image_processor, transform=None):
        self.labels = []
        self.file_paths = []
        self.image_processor = image_processor
        for label in file_names.keys():
            for name in file_names[label]:
                full_path = os.path.join(file_path, label, name)
                self.file_paths.append(full_path)
                self.labels.append(label2id[label])
        
    def __len__(self):
        return len(self.file_paths)
    
    def __getitem__(self, idx):
        vid_container = av.open(self.file_paths[idx])
        vid_indices = sample_frame_indices(clip_len=32, frame_sample_rate=1, seg_len=vid_container.streams.video[0].frames)
        vid_read = read_video_pyav(container=vid_container, indices=vid_indices)
        vid_input = self.image_processor(list(vid_read), return_tensors="pt")
        vid_input['pixel_values'] = vid_input['pixel_values'].squeeze(0)
        
        return {'video' : vid_input, 'class' : torch.tensor(self.labels[idx], dtype=torch.long)}

In [5]:
def split_dataset(filenames, test_size=0.2, random_state=20):
    test = {}
    train = {}
    for label in filenames.keys():
        train_names, test_names = train_test_split(filenames[label], test_size=test_size, random_state=random_state)
        test[label] = test_names
        train[label] = train_names
    return train, test

# Preparing the dataset and splititng into training and test datasets
train_vid_names, test_vid_names = split_dataset(filenames, random_state=8)

In [6]:
class Decoder(nn.Module):
    def __init__(self, input_dim, hidden_size, num_labels):
        super(Decoder, self).__init__()
        self.hidden_size = hidden_size
        self.fc1 = nn.Linear(input_dim, hidden_size)
        self.fc2 = nn.Linear(hidden_size, num_labels)
        self.drop = nn.Dropout(p = 0.2)
    def forward(self, x):
        # x = x.view(x.size(0), -1)
        x = torch.sigmoid(self.fc1(x))
        x = F.softmax(self.fc2(x), dim=1)
        return x

In [7]:
# USING AN ENCODER AND A DECODER

# image_processor = VivitImageProcessor.from_pretrained("google/vivit-b-16x2-kinetics400")
# encoder = VivitModel.from_pretrained("google/vivit-b-16x2-kinetics400")
# decoder = Decoder(768, 3137, 12)
# decoder = Decoder(3137 * 768, 768, 12)

# dataset = HandwashingDataset(data_dir, train_vid_names, image_processor)
# data_loader = DataLoader(dataset, batch_size = 1, shuffle=True)

# num_epochs = 8

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# encoder.to(device)
# decoder.to(device)
# decoder.train()

# criterion = nn.CrossEntropyLoss().cuda() if torch.cuda.is_available else nn.CrossEntropyLoss()

# params = list(decoder.parameters())
# optimizer = torch.optim.Adam(params)
# optimizer.zero_grad()

In [8]:
config = VivitConfig()
config.num_labels = len(classes)
model = VivitForVideoClassification(config)
# model.load_state_dict(torch.load("cur_model_weights.pkl", weights_only=True))

criterion = nn.CrossEntropyLoss().cuda() if torch.cuda.is_available else nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

image_processor = VivitImageProcessor(rescale_factor=1/255.0,offset=False,size={"height": 224, "width": 224})
dataset = HandwashingDataset(data_dir, train_vid_names, image_processor)
data_loader = DataLoader(dataset, batch_size = 1, shuffle=True)
num_epochs = 8

In [9]:
model.to(device)
model.train()

optimizer = torch.optim.Adam(model.parameters())
optimizer.zero_grad()

for epoch in range(1, num_epochs+1):
    running_loss = 0.0 
    correct_predictions = 0
    total_samples = 0
    for batch in tqdm(data_loader, desc=f"Epoch {epoch}/{num_epochs}"):
        video = batch['video'].to(device)
        labels = batch['class'].to(device)
        
        features = model(**video)
        outputs = F.softmax(features.logits, dim=1)
        
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        preds = outputs.argmax(dim=-1)  # Get predicted class
        correct_predictions += (preds == labels).sum().item()
        total_samples += labels.size(0)
        
    epoch_loss = running_loss / total_samples
    epoch_acc = correct_predictions / total_samples
    print(f'Epoch [{epoch}/{num_epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}')

Epoch 1/8:   0%|          | 0/240 [00:00<?, ?it/s]/opt/conda/lib/python3.10/site-packages/transformers/feature_extraction_utils.py:142: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /usr/local/src/pytorch/torch/csrc/utils/tensor_new.cpp:278.)
  return torch.tensor(value)
Epoch 1/8: 100%|██████████| 240/240 [12:16<00:00,  3.07s/it]


Epoch [1/8], Loss: 2.5484, Accuracy: 0.0667


Epoch 2/8: 100%|██████████| 240/240 [12:05<00:00,  3.02s/it]


Epoch [2/8], Loss: 2.5354, Accuracy: 0.0833


Epoch 3/8: 100%|██████████| 240/240 [12:15<00:00,  3.07s/it]


...

Epoch [6/8], Loss: 2.5354, Accuracy: 0.0833


Epoch 7/8: 100%|██████████| 240/240 [12:09<00:00,  3.04s/it]


Epoch [7/8], Loss: 2.5354, Accuracy: 0.0833


Epoch 8/8: 100%|██████████| 240/240 [12:08<00:00,  3.04s/it]

Epoch [8/8], Loss: 2.5354, Accuracy: 0.0833


In [10]:
# torch.save(model, "cur_model.pkl")
torch.save(model.state_dict(), "cur_model_weights.pkl")

<a href="cur_model.pkl"> Download File </a>